# Phase1: 非AIのベースライン — 「AIなし」でどこまで読めるか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: AI（機械学習）を使わず、色や形のルールだけで板を見つけ、文字を読んでみる回です。
> - **たとえ話**: 定規と見本帳だけで、板を探して文字を読む。簡単な画像ならできるが、難しい画像には弱い。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-1・2-2（枠とIoU）・2-3（CER）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 15分 ＋ 実行 2分

## このプロジェクトのゴール（おさらい）
駐車場のナンバー読み取り精算システムの「ミニ版」を作り、**AIはシステムの一部品**であることを確かめます。
前回（Phase0）は、練習用の**合成画像と正解**を作りました。

## このNotebookのゴール
**機械学習を使わず、人間が書いたルールだけで**、ナンバー板を「見つけ」「読む」システムを作り、**どこまでできて、どこで壊れるか**を測ります。

### なぜ「AIなし」を先に作るのか
- **AIを使う価値を、実感するため。** AIなしでほぼ完璧にできるなら、AIは要りません。どこまで壊れるかを知って、はじめて「AIが何を解決するか」が分かります。
- **比較の基準（ベースライン）になるため。** あとでAIを作ったとき、「AIなしと比べてどれだけ良いか」を測れます。
- **システムの全体像が、いちばん見やすいため。** 古典的な手法は、各段階で何をしているかが目で見えます。

## 古典的な画像処理とは
**人間が「板はこういうものだ」というルールを、手で書く**方法です。たとえば次のようなルールです。
- 板は**白っぽくて明るい**（→ 明るくて色の薄い領域を探す）
- 板は**横長**で、縦横比が約2:1
- 板の中に**緑色の文字**がある
- 文字は**決まった位置**に並ぶ（→ 位置ごとに、見本の文字と見比べる）

AI（機械学習）は、これらのルールを人間が書く代わりに、**例から自分で見つけます**（Phase2以降）。

## このNotebookでやること
1. **検出**を1段階ずつ見る: 明るく薄い色の領域 → すき間を埋める → つながった塊を数える → 板らしさで採点
2. **文字の読み取り**を1段階ずつ見る: 板をまっすぐに直す → 文字のインク画像 → 見本と見比べる
3. 難易度別に**測る**: 検出だけ／読み取りだけ／全体
4. **失敗の原因**を分析する（最重要）

## キーワード
| 用語 | 意味 |
|---|---|
| **古典的な画像処理** | 機械学習を使わず、人間が書いたルールで画像を処理する方法 |
| **マスク（2値画像）** | 条件に合う画素を白、合わない画素を黒にした画像 |
| **クロージング／オープニング** | 白い領域を膨らませて縮める（すき間を埋める）／縮めて膨らませる（細い橋を切る） |
| **連結成分** | つながった白い画素のひとかたまり |
| **透視変換** | 傾いた・ゆがんだ板を、長方形にまっすぐ直す処理 |
| **テンプレートマッチ** | 見本の文字（テンプレート）を重ね、どれにいちばん似ているかを調べる方法 |
| **正規化相互相関（類似度）** | 2つの画像がどれくらい似ているか（−1〜1。1に近いほど似ている） |
| **IoU** | 予測の枠と正解の枠の重なり（1に近いほど一致） |
| **CER（文字誤り率）** | 文字の間違いの割合。0なら全部正解 |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from plate_synth import generate_sample, split_seed
from plate_synth.plate import nominal_cells
from plate_synth.viz import show_grid
from classical import detect_plate, read_plate, read_text, rectify
from classical.detect import PARAMS, candidates
from classical.ocr import cell_scores, ink_map
from plate_eval import cer, exact_match, iou

%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())

上のセルは、道具の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

## 1. 検出 — 板を見つける

処理の対象は、Phase0 の評価用（val）と同じシードで作った画像です。まず Easy の1枚を使います。

In [ ]:
# 検証セット(val)のEasyの1枚目と、同じ画像を再生成する（シードが同じなので、同じ画像になる）
d = "easy"
s = generate_sample(split_seed("val", d, 1), d)
img, label = s.image, s.label
print("正解の文字:", label["text"], "／ 正解の枠:", label["bbox"])
img

### 手順① 「明るくて、色が薄い」画素を探す
板は**白っぽい**（明るくて、彩度が低い）ので、その条件に合う画素だけを白、他を黒にした**マスク**を作ります。

- **明るさ V**: 画素のRGBの最大値。「画像の最も明るい部分の7割以上」を明るいとする（暗い画像でも使えるよう、**相対的**に決める）。
- **彩度 S**: 色の濃さ。薄い（白・灰色に近い）ものだけを残す。
- この**しきい値（7割、0.25など）は人間が決めた数字**です。ここが古典的手法の弱点になります（あとで確認）。

### 手順② すき間を埋める（クロージング、穴埋め）
板の中の**緑の文字は「明るくない」ので、マスクでは穴**になります。そのままでは、板が穴だらけの塊になるので、
- **クロージング**: 白を少し膨らませてから縮め、細いすき間を埋める
- **穴埋め**: 外につながっていない黒い部分（文字の穴）を、白で埋める

### 手順③ つながった塊を数える（連結成分）
白い画素の**ひとかたまり**ごとに番号を付けます。次のセルで、①→②→③の様子を並べます。

In [ ]:
cands, stages = candidates(img, PARAMS)

fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
axes[0].imshow(img); axes[0].set_title("元の画像")
axes[1].imshow(stages["mask"], cmap="gray"); axes[1].set_title("① 明るく色の薄い画素")
axes[2].imshow(stages["closed"], cmap="gray"); axes[2].set_title("② クロージング後")
axes[3].imshow(stages["filled"], cmap="gray"); axes[3].set_title("② 穴埋め＋オープニング後")
lab = stages["labels"]
axes[4].imshow(np.ma.masked_where(lab == 0, lab), cmap="tab10"); axes[4].set_title(f"③ 連結成分（{lab.max()}個）")
for a in axes: a.axis("off")
fig.tight_layout();

**見るポイント**
- **①**: 板だけでなく、**ほかの明るい物**（背景の白っぽい部分や看板）も白くなっていませんか？ 板の中は**文字の部分が穴**になっていますか？
- **②**: 板の穴が埋まって、**ひとつの塊**になりましたか？ オープニングで、**細い橋でつながっていた別の物**が切り離されていますか？
- **③**: 塊ごとに色が付きます。板は1つの塊になっていれば成功です。

### 手順④ 候補を「板らしさ」で採点する
塊ごとに、次の4つを調べます。**全部が基準を満たしたものだけ**が板の候補になります。

| 項目 | 意味 | 基準（人間が決めた数字） |
|---|---|---|
| **縦横比** | 長辺÷短辺 | 1.3〜3.2（板は約2:1） |
| **埋まり率** | 塊の面積 ÷ それを囲む回転した長方形の面積 | 0.6以上（長方形に近い形か） |
| **インク率** | 塊の中の「穴（文字）」の割合 | 0.04〜0.6（文字があるか） |
| **緑がかり** | 文字（穴の部分）が緑色か | 0.05以上（白い看板の黒い棒と区別する） |

In [ ]:
df = pd.DataFrame([{k: (round(v, 3) if isinstance(v, float) else v) for k, v in c.items() if k not in ("corners",)} for c in cands])
print("候補（連結成分のうち、大きさの条件を満たしたもの）:")
display(df)

det = detect_plate(img)
if det is None:
    print("→ 検出失敗（板が見つからなかった）")
else:
    print(f"→ 検出成功。IoU = {iou(det.bbox, label['bbox']):.3f}（1に近いほど、正解の枠と一致）")

In [ ]:
# 検出した四隅（赤）と、正解の枠（緑）を重ねて見る
fig, ax = plt.subplots(figsize=(5, 3.8))
ax.imshow(img); ax.axis("off")
x, y, w, h = label["bbox"]
ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec="lime", lw=2, label="正解の枠"))
if det is not None:
    poly = np.vstack([det.corners, det.corners[:1]])
    ax.plot(poly[:, 0], poly[:, 1], "r-", lw=2, label="検出した四隅")
ax.legend(loc="lower right", fontsize=8);

## 2. 文字の読み取り — 見本と見比べる

検出した四隅から、板を読みます。

### 手順① 板をまっすぐに直す（透視変換）
傾いた板を、**長方形（384×192）にまっすぐ直し**ます。これで、文字の位置が決まった場所になります。

### 手順② 「インクの濃さ」の画像にする
明るさの違い（暗い画像でも）に左右されないよう、画像を「**背景=0、文字=1**」のインク画像にします（明るい部分と暗い部分の基準を、画像ごとに決める）。縁取りとボルト穴は無視します。

### 手順③ 文字の「セル」ごとに、見本と見比べる
ここが古典的手法の特徴です。**板の書式を、人間が手で書いて持っています。**

```
 上段:  [地名 2文字]        [分類番号 3文字]
 下段:  [かな 1文字]  [一連番号 4〜5文字]
```

文字が並ぶ位置（**セル**）は決まっているので、各セルで**見本の文字**（地名なら5種類の漢字、数字なら0〜9など）を重ね、**いちばん似ているもの**を選びます。
少しずれていても大丈夫なよう、見本を前後左右に3ピクセルずつずらして、最もよく合う位置で比べます。

次のセルで、①〜③の様子を見ます。

In [ ]:
plate = rectify(img, det.corners if det is not None else label["corners"])     # ① まっすぐに直す
ink = ink_map(plate)                                                              # ② インク画像（半分の解像度）

fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
axes[0].imshow(plate); axes[0].set_title("① まっすぐに直した板")
axes[1].imshow(ink, cmap="gray_r"); axes[1].set_title("② インクの濃さ（黒=文字）")
# ③ 標準のセル配置（一連番号が「ハイフンあり」の場合）を重ねる
axes[2].imshow(plate); axes[2].set_title("③ 文字のセル（赤い枠）")
for c in nominal_cells(serial_has_hyphen=True):
    w, h = c["adv"], c["size"] * 1.1
    axes[2].add_patch(plt.Rectangle((c["cx"] - w / 2, c["cy"] - h / 2), w, h, fill=False, ec="red", lw=1))
for a in axes: a.axis("off")
fig.tight_layout();

次に、**1つのセル**で、どの文字がどれくらい似ているかを見ます。ここでは、**上段の左の漢字（地名の1文字目）**を取り上げます。
類似度は−1〜1で、**1に近いほど見本とよく一致**しています。

In [ ]:
cells = nominal_cells(serial_has_hyphen=False)
cell = cells[0]                                   # 地名の1文字目
chars, scores = cell_scores(ink, cell)
order = np.argsort(-scores)
print(f"正解の地名: {label['region']}（1文字目は「{label['region'][0]}」）")
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.bar([chars[i] for i in order], [scores[i] for i in order], color=["C3"] + ["gray"] * (len(chars) - 1))
ax.set(ylim=(min(0, scores.min()), 1), ylabel="類似度", title="地名の1文字目: 見本との類似度（赤=最も似ている）")
ax.grid(axis="y", alpha=0.3);

**見るポイント**: 1位の文字と、2位の文字の**差**はどれくらいですか？ 差が大きければ**自信を持って**答えられ、小さければ**迷っている**状態です。
（この「類似度」は、あとの Phase4 で「自信がないときは要確認にする」ための確信度の元になります。）

### 手順④ 一連番号は「4セル」か「5セル」か
一連番号は、`12-34`（5文字、ハイフンあり）か、`・・12`（4文字）のどちらかで、**文字数が違います**。古典的手法は、**両方の配置を試して、全体として似ているほうを採用**します。

次のセルで、全部をつなげて読みます（**検出 → まっすぐに直す → 読む**）。

In [ ]:
res = read_plate(img)
print("正解:", label["text"])
print("読み:", res.text, "  ← 完全一致:", exact_match(res.text, label["text"]))
print("文字ごとの類似度（小さいほど自信がない）:", np.round(res.ocr.char_scores, 2))
print(f"板全体の確信度の目安（最小の類似度）: {res.ocr.confidence:.2f}")

## 3. 難易度別に測る

1枚だけでは、何も言えません。検証セット（val）の画像で、難易度別に**3つの見方**で測ります。

| 見方 | 内容 | 何が分かるか |
|---|---|---|
| **検出だけ** | 板の位置が合っているか（IoU ≥ 0.5 を成功とする） | 板を見つける力 |
| **OCRだけ** | **正解の四隅**で板をまっすぐにしてから読む | 文字を読む力そのもの（検出の失敗を除く） |
| **全体** | 検出 → 読み取りを通す | システムとしての力 |

この差を取ることで、**「検出で失敗したのか、読み取りで失敗したのか」**を切り分けられます。

> 📌 検出のしきい値は、val の**先頭100枚**を見ながら人間が調整しました。そのため、ここでの評価は**101枚目以降**を使います。**test は最後に1回だけ**使うので、ここでは使いません。
> 📌 下は短時間のため **各難易度60枚**です。全400枚の結果は、ターミナルで `python src/eval_baseline.py` を実行すると得られます（約20秒）。

In [ ]:
from plate_synth import load_labels
EVAL = ROOT / "data" / "eval"
if not EVAL.exists():
    raise SystemExit("評価セットが未作成です。ターミナルで  .venv/bin/python src/make_eval_sets.py  を実行してください。")

N = 60            # 各難易度の枚数（101枚目から）
rows = []
for d in ["easy", "medium", "hard"]:
    for lab in load_labels(EVAL / "val" / d / "labels.jsonl")[100:100 + N]:
        im = Image.open(EVAL / "val" / d / "images" / lab["image"]).convert("RGB")
        r = read_plate(im)
        o = read_text(rectify(im, lab["corners"]))                      # OCRだけ（正解の四隅）
        rows.append(dict(難易度=d, iou=iou(r.detection.bbox, lab["bbox"]) if r.detection else 0.0,
                         detected=r.detection is not None,
                         ocr_only=exact_match(o.text, lab["text"]), exact=bool(r.text) and exact_match(r.text, lab["text"]),
                         cer=cer(r.text, lab["text"]) if r.text else 1.0))
res = pd.DataFrame(rows)
summary = res.groupby("難易度", sort=False).agg(
    検出率=("iou", lambda x: (x >= 0.5).mean()), OCRだけ_完全一致=("ocr_only", "mean"),
    全体_完全一致=("exact", "mean"), 全体_CER=("cer", "mean")).round(3)
summary

**表の見方**: 値は割合（0.9 = 90%）。CER は小さいほど良い。

## 4. 失敗の原因を分析する（最重要）

次のグラフは、**全体で失敗した画像**を、原因別に分けたものです。

| 原因 | 意味 |
|---|---|
| **成功** | 完全一致 |
| **未検出** | 板が見つからなかった |
| **検出ずれ** | 板は見つけたが、位置が大きくずれた（IoU < 0.5） |
| **読み間違い** | 板の位置は合っている（IoU ≥ 0.5）が、文字を間違えた |

In [ ]:
def cause(r):
    if r.exact: return "成功"
    if not r.detected: return "未検出"
    if r.iou < 0.5: return "検出ずれ"
    return "読み間違い"
res["原因"] = res.apply(cause, axis=1)
order = ["成功", "未検出", "検出ずれ", "読み間違い"]
tab = pd.crosstab(res["難易度"], res["原因"], normalize="index").reindex(columns=order, fill_value=0).loc[["easy", "medium", "hard"]]
tab.plot(kind="bar", stacked=True, figsize=(7, 3.6), color=["#4c9f70", "#999999", "#e6a23c", "#d9534f"])
plt.ylabel("割合"); plt.title("全体の結果の内訳（難易度別）"); plt.xticks(rotation=0); plt.legend(loc="upper right", fontsize=8);
tab.round(3)

**見るポイント**
- **難易度が上がるほど、成功が急に減り、「未検出」が増えます。** 条件が変わると、人間が決めたしきい値が合わなくなるためです。
- **Easy でも、OCRだけ（正解の四隅）ではほぼ満点なのに、全体では失敗が出ます。** 検出が「だいたい合っている」だけでは足りない、ということです。次のグラフで確かめます。

### 検出の位置ずれは、読み取りにどう響くか
検出の**IoU**（枠の一致度）ごとに、その後の**読み取りの成功率**を見ます。

In [ ]:
res["IoUの範囲"] = pd.cut(res.iou, [-0.01, 0.0, 0.5, 0.7, 0.8, 0.9, 1.0], labels=["未検出", "<0.5", "0.5-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0"])
g = res.groupby("IoUの範囲", observed=True).agg(枚数=("exact", "size"), 読み取り成功率=("exact", "mean")).round(3)
display(g)
ax = g["読み取り成功率"].plot(kind="bar", figsize=(6, 3.2), color="C0")
ax.set(ylabel="完全一致の割合", ylim=(0, 1), title="検出の一致度(IoU)と、読み取りの成功率")
plt.xticks(rotation=0);

**重要な発見**
- **IoUが0.9以上でないと、読み取りはほとんど成功しません。** 「IoU 0.5以上なら検出成功」という一般的な基準では、**OCRにとっては足りません。**
- 理由: 板の四隅が少しずれると、まっすぐに直したときに**文字の位置がずれ**、決まった位置のセルと合わなくなる。
- → **誤りは後ろの段階に伝わります**（検出の誤差 → 補正の誤差 → 読み取りの誤り）。システム全体で見ないと、本当の性能は分かりません。

### 失敗例を目で見る
全400枚の評価（`python src/eval_baseline.py`）を実行すると、失敗例の画像が `outputs/baseline/` に出ます（緑=正解の枠、赤=検出した枠）。あれば表示します。

In [ ]:
from IPython.display import Image as IPImage, display
shown = 0
for name in ["failures_det_easy.png", "failures_ocr_medium.png", "failures_det_hard.png"]:
    f = ROOT / "outputs" / "baseline" / name
    if f.exists():
        display(IPImage(filename=str(f))); shown += 1
if not shown:
    print("失敗例の画像が未作成です。ターミナルで  .venv/bin/python src/eval_baseline.py  を実行してください。")

## 5. なぜ古典的手法は壊れるのか（まとめ）

| 弱点 | 説明 | この実験での現れ方 |
|---|---|---|
| **しきい値が人間の決めた数字** | 「明るさ7割」「彩度0.25」などが、条件に合わないと機能しない | 暗い・逆光の画像で、板が検出されない |
| **似た別の物に弱い** | 背景の明るい部分が板と同じ条件を満たす | 背景とつながって、形が崩れる |
| **書式を手で書く** | 文字の位置・種類を人間がコードに書く。別の種別・配置には作り直し | 事業用（緑地）や別の配置には、そのままでは使えない |
| **劣化に弱い** | ぼけ・ノイズ・低解像度で、小さな漢字の細部が失われる | Medium で「浜」と「縄」などを取り違える |
| **誤りが後ろに伝わる** | 検出の位置ずれが、補正・読み取りの失敗につながる | IoUが低いと、読み取りがほぼ失敗 |

### 逆に、古典的手法の良い点
- **仕組みが目で見える**（どこで失敗したか分かりやすい）。
- **きれいな条件では、ほぼ完璧**（OCRだけ・Easyは満点）。学習データも要らない。
- **非常に速い**（1枚あたり数十ミリ秒）。

> 💡 **次の Phase2〜3 で、AI（機械学習）に置き換えると、何が良くなるか？** ルールを人間が書く代わりに、**例（合成画像）から学習**します。
> 暗さ・ぼけ・傾きが混ざった条件でも、学習データに含めれば対応できる可能性があります。**この基準（ベースライン）より、どれだけ良くなるか**を、あとで測ります。

## 🧪 試してみよう

### 演習1: 検出のしきい値を変えてみる
**やること**: 下のセルの **`v_ratio = 0.7`**（「明るい」の基準）を **`0.9`**、**`0.5`** に書き換えて実行します。
- **見るポイント**: 3つの難易度の検出率は、どう変わりますか？ Easy で良くなる値が、Hard では悪化しませんか？
- **分かること**: **1つのしきい値で、全部の条件に対応するのは難しい**。条件ごとに調整が必要になる（これが古典的手法の限界）。

In [ ]:
v_ratio = 0.7        # ← 0.9 や 0.5 に書き換えて実行してください
rows = []
for d in ["easy", "medium", "hard"]:
    hit = 0
    for i in range(100, 140):                                        # 検証セットの40枚
        s = generate_sample(split_seed("val", d, i), d)
        det = detect_plate(s.image, dict(v_ratio=v_ratio))
        hit += det is not None and iou(det.bbox, s.label["bbox"]) >= 0.5
    rows.append({"難易度": d, "検出率": hit / 40})
pd.DataFrame(rows).set_index("難易度")

### 演習2: 劣化を強くして、読み取りの限界を探す
**やること**: 下のセルの **`noise_sigma = 10.0`**（ノイズの強さ）を **`20.0`**、**`40.0`** に書き換えて実行します。次に **`blur_sigma`** でも試します（`overrides` の `noise_sigma=noise_sigma` を `blur_sigma=blur_sigma` に変える）。
- 「OCRだけ」（正解の四隅で直す）ので、**検出は関係なく、読み取りの力だけ**が分かります。
- **見るポイント**: 何から先に、読み取り成功率が急に下がりますか？
- **分かること**: 劣化には**限界**があり、その手前で人間にも読みにくくなる。**限界の位置を知る**ことが、あとの「要確認に回す」設計の出発点になります。

In [ ]:
from plate_synth.difficulty import CLEAN
noise_sigma = 10.0   # ← 20.0 や 40.0 に書き換えて実行してください
ok = 0
for i in range(40):
    smp = generate_sample(split_seed("val", "easy", i), "easy", overrides={**CLEAN, "bg_type": "wall", "scale": 0.5, "noise_sigma": noise_sigma})
    ok += exact_match(read_text(rectify(smp.image, smp.label["corners"])).text, smp.label["text"])
print(f"ノイズ σ={noise_sigma}: OCRだけの完全一致 {ok}/40 = {ok / 40:.2f}")
smp.image

### 演習3: 別の種別の板で試す（事業用）
**やること**: 下のセルを実行します。**事業用**（緑地に白文字）の板を、同じ古典的手法で検出・読み取りします。
- **見るポイント**: 検出率・読み取りは、自家用（白地に緑文字）と比べてどうなりますか？
- **分かること**: 古典的手法は、「**白い板＋緑の文字**」という前提を**コードに書いている**ので、前提が変わると使えません。**種別ごとに作り直す必要**があります。AI（学習）なら、例を増やせば対応できる可能性があります。

In [ ]:
hit = ok = 0
for i in range(30):
    smp = generate_sample(split_seed("val", "easy", i), "easy", plate_type="business")
    det = detect_plate(smp.image)
    hit += det is not None and iou(det.bbox, smp.label["bbox"]) >= 0.5
    ok += bool(read_plate(smp.image).text) and exact_match(read_plate(smp.image).text, smp.label["text"])
print(f"事業用（Easy 30枚）: 検出率 {hit / 30:.2f} ／ 全体の完全一致 {ok / 30:.2f}")
smp.image

### 考えてみよう（コード不要）
1. 古典的手法を**さらに磨けば**、Hard でもほぼ読めるようになるでしょうか？ そのとき、**人間が書くルール**は、どれくらい増えそうですか？
2. 「検出のIoUが0.9以上でないと読めない」という発見は、**検出と読み取りを別々に評価するだけでは見つからない**ものです。なぜ、システム全体で見ることが大切なのでしょう？
3. 「Medium で全体の完全一致が1割」のシステムを、そのまま駐車場に置いたらどうなりますか？ 次の Phase4 の「**要確認**」は、これをどう救えそうですか？

---

## ✅ まとめ

- **古典的な画像処理（機械学習なし）**は、人間が書いたルール（明るさ・形・色・文字の位置）で、板を探し、文字を読む。**仕組みが目で見え、きれいな条件ではほぼ完璧**で、速い。
- ただし、**条件が変わる（暗い・ぼけ・傾き・汚れ・別の種別）と、急に壊れる。** しきい値や書式を、人間が条件ごとに調整し直す必要がある。
- **誤りは後ろの段階に伝わる。** 検出のIoUが0.9以上でないと、読み取りはほとんど成功しない。システム全体で測ることが大切。
- 結果（val の101枚目以降、各400枚）は、`python src/eval_baseline.py` の出力と [MANUAL.md](../MANUAL.md) の実験メモを参照。
- これが**AIとの比較の基準（ベースライン）**になる。

➡ 次は **Phase2: 検出（物体検出）**。ルールを手で書く代わりに、**例から学習するAI**で板の位置を見つけます。